### Exploração Inicial de Dados

### Sumário
- Resumo Database.
- Percentual de tempo por categoria.
- Percentual de tempo por categoria(Apenas momentos com emissão).
- Percentual de minutos com emissão por hora do dia.
- Minutos com emissão pela variável tempo.
- Minutos de emissão por dia.
- Análise de Outlier.
- Boxplot Análise.
- Valores Faltantes.
- Principal Componente Analysis.

### Resumo Database

In [0]:
%pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd #Biblioteca focada em extração, transformação e carga de dados (ETL)
import matplotlib.pyplot as plt #Biblioteca para produção de gráficos
from datetime import datetime #Manipulação de datas
import seaborn as sns

In [0]:

df = pd.read_csv('/Volumes/catalog_samarco/storage_datalake/dbw-otimizacao/pluma_multiclasse_5krows/df_reduzida.csv')
df

Ações:
- Criar cópia dataframe.
- Remover colunas.
- Criação coluna Dia/Noite, filtro apenas noite.
- Criação coluna Tempo.




In [0]:
ds = df.copy()
ds

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys

import os

from src.feature_utils import adicionar_periodo_dia



sys.path.append(os.path.abspath('..'))

ds = (

    adicionar_periodo_dia(ds, coluna_timestamp_str='t')

    .rename(columns={'periodo': 'Dia/Noite'})

    .loc[lambda x: x['Dia/Noite'] == 'Dia']

    .assign(

        Tempo=lambda x: (pd.to_timedelta(x['Hora.1']).dt.total_seconds() / 60).astype(int)

    )

    .drop(columns=["Unnamed: 0","Amostragem","Dia/Noite", "t", "n", "Hora.1",'Ano'])

)


ds




In [0]:
X = ds.copy()

## Percentual de tempo por categoria

In [0]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
# (Assumindo que 'pandas' e 'dx' já estão carregados)

# Seus cálculos de percentual (estão corretos)
contagem_minutos = ds['Legenda Y'].value_counts()
total_minutos = len(ds)
percentual = (contagem_minutos / total_minutos) * 100

# Cor da barra
cor_barra = '#004170' 

# Cria a figura e os eixos
fig, ax = plt.subplots(figsize=(16, 9)) 

barras = ax.barh(percentual.index, percentual.values, color=cor_barra) 
ax.set_title("Percentual de tempo por categoria de emissão", fontsize=16, pad=20) 
ax.invert_yaxis() 

for bar in barras: 
    width = bar.get_width() 
    rotulo = f'{width:.1f}%' 
    
    ax.text(
        width + 0.5, 
        bar.get_y() + bar.get_height()/2, 
        rotulo, 
        va='center', 
        ha='left',   
        fontsize=12
    )

plt.gca().spines['right'].set_visible(False) 
plt.gca().spines['top'].set_visible(False) 
plt.grid(axis='x', linestyle='--', alpha=0.5) 
plt.tight_layout()



### Percentual do tempo por categoria (Apenas momentos com emissão)

In [0]:
ds = ds[ds['Y'] != 0]
contagem_minutos = ds['Legenda Y'].value_counts() #Contagem dos minutos por categoria
total_minutos = len(ds) #Contagem do total de minutos
percentual = (contagem_minutos / total_minutos) * 100 #Cálculo dos percentuais por categoria

cor_barra = '#004170' #Código HEX para o azul escuro da Samarco
fig, ax = plt.subplots(figsize=(16, 9)) #Para que a imagem tenha a mesma proporção de um slide Powerpoint
barras = ax.barh(percentual.index, percentual.values, color=cor_barra) #Cria o gráfico
ax.set_title("Percentual de tempo por categoria (Apenas momentos com emissão)", fontsize=16, pad=20) #Adiciona o título do gráfico
ax.invert_yaxis() #inverte o eixo Y para que a maior categoria fique me primeiro
for bar in barras: #Roda o código abaixo para cada barra do gráfico
    percentual = bar.get_width() #A largura da barra é o percentual
    rotulo = f'{percentual:.1f}%' #Formata o rótulo para que só tenha uma casa decimal
    # Posicionar o rótulo:
    # x: Posição horizontal, um pouco à direita da extremidade da barra (externo)
    # y: Centro vertical da barra (bar.get_y() + bar.get_height()/2)
    ax.text(
        percentual + 0.5, # Ajuste o '0.5' para controlar a distância da barra (extremidade direita)
        bar.get_y() + bar.get_height()/2, 
        rotulo, 
        va='center', # Alinhamento vertical: centralizado
        ha='left',   # Alinhamento horizontal: à esquerda (para que o texto comece após a barra)
        fontsize=12
    )
plt.gca().spines['right'].set_visible(False) # Remove a borda direita
plt.gca().spines['top'].set_visible(False) # Remove a borda esquerda
plt.grid(axis='x', linestyle='--', alpha=0.5) # Adiciona grades
plt.tight_layout()
barras

### Percentual de Minutos por hora do dia

In [0]:
ds['Emissão'] = (ds['Y'] != 0).astype(int) #Cria uma coluna binária onde os minutos com emissão recebem 1 e o restante zero.
contagem_por_hora = ds.groupby('Hora')['Emissão'].sum() #Agrupa a tabela usando a coluna Horas como totalizadora da coluna Emissão
total_emissao = ds['Emissão'].sum() #Calcula o número total de minutos com emissão.
percentuais_por_hora = (contagem_por_hora / total_emissao) * 100 #Calcula os percentuais por hora

categorias_horas = [str(h) for h in percentuais_por_hora.index]
valores_percentuais = percentuais_por_hora.values
cor_barra = '#004170'
titulo_grafico = "Percentual de minutos com emissão por hora do dia"
fig, ax = plt.subplots(figsize=(16, 9))
barras = ax.bar(categorias_horas, valores_percentuais, color=cor_barra) #Plota o gráfico
ax.set_title(titulo_grafico, fontsize=20, pad=30)
for bar in barras:
    percentual = bar.get_height() # Em barras verticais, a altura é o valor
    rotulo = f'{percentual:.1f}%'
    
    # Posicionar o rótulo (acima da barra e centralizado)
    ax.text(
        bar.get_x() + bar.get_width() / 2, # Centro horizontal da barra
        bar.get_height() + 0.5,           # Um pouco acima da barra (ajuste o 0.5 conforme necessário)
        rotulo, 
        ha='center', # Alinhamento horizontal: centralizado
        va='bottom', # Alinhamento vertical: inferior (o texto começa acima do ponto)
        fontsize=12 
    )
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
barras

### Percentual de minutos com emissão pela variável Tempo

In [0]:
emissao_diaria = ds.groupby('Tempo')['Emissão'].sum()
fig, ax = plt.subplots(figsize=(16, 9))
ax.plot(emissao_diaria.index, emissao_diaria.values, 
        color='#004170', 
        marker='o', # Ponto por dia
        linestyle='-', # Linha conectando os pontos
        linewidth=2, 
        markersize=5) 
ax.set_title("Minutos com emissão pela variável Tempo", fontsize=20, pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show

### Minutos com emissão por dia

In [0]:
ds['Data'] = pd.to_datetime(ds['Data']).dt.date
emissao_diaria = ds.groupby('Data')['Emissão'].sum()
fig, ax = plt.subplots(figsize=(16, 9))
ax.plot(emissao_diaria.index, emissao_diaria.values, 
        color='#004170', 
        marker='o', # Ponto por dia
        linestyle='-', # Linha conectando os pontos
        linewidth=2, 
        markersize=5) 
ax.set_title("Minutos com emissão por dia", fontsize=20, pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show

### Boxplot Análise
Aqui iremos utilizar o boxplot para entender como está a distribuição dos dados nas variáveis de processo. Inicialmente iremos analisar todas as variáveis conjuntamente, para tal iremos normalizar os dados e em seguida separadamente.
### Objetivo:
- Entender a distruibuição de dados de cada variáveil.
- Entender a distruibuição separada olhando para Outliers nos dados.

In [0]:
import sys
import os

variaveis_processo = [coluna for coluna in X.columns if '[US3]' in coluna or '[US4]' in coluna or 'Velocidade do vento (Medida)' in coluna]


sys.path.append(os.path.abspath('..'))
%load_ext autoreload
%autoreload 2
from src.feature_utils import normalizar_dados, plotar_boxplots,MedidasEstatisticas,qtdColunasNulas


X = normalizar_dados(X)
part1 = variaveis_processo[0:(len(variaveis_processo)//2)]
part2 = variaveis_processo[(len(variaveis_processo)//2):]
plotar_boxplots(X[part1])
plotar_boxplots(X[part2])

Com isso podemos concluir que é necessário fazer alguns tratamentos de dados nas variáveis devido a alta presença de outlier nos dados. Como exemplos podemos ver alta presença de outlier nas seguintes variáveis:
- [US4] Umidade.
- [US4] Teor de Carvão.
- [US4] Vazão na Chaminé.
- [US4] Concentração de Nox.
- [US4] Pressão na pré-queima.
- [US4] Vazão de Gás.
- [US4] Temperatura na caixa de vento 22.
- [US3] Umidade.
- [US3] Teor de Carvão.


entre outras..


Agora vamos olhar para algumas variáveis excluviamente para entender a quantide de outliers possuem tais variáveis:

In [0]:


sys.path.append(os.path.abspath('..'))
%load_ext autoreload
%autoreload 2
from src.feature_utils import adicionar_caixa_estatisticas

# Seleciona o nome da coluna que será plotada
nome_da_coluna = variaveis_processo[0]

# Cria a figura com APENAS 1 subplot (eixo)
fig, ax = plt.subplots(figsize=(19, 8))

# --- Plot Único ---
# A coluna específica do seu DataFrame é passada aqui
sns.boxplot(data=X[nome_da_coluna], ax=ax, orient='v')
ax.set_title(f'Boxplot e Análise de Outliers para "{nome_da_coluna}"')
ax.set_ylabel('Distribuição dos dados')
ax.set_xlabel(nome_da_coluna)

# Chama a função para adicionar a caixa de estatísticas no plot
# A série de dados da sua coluna é passada como argumento
adicionar_caixa_estatisticas(ax, X[nome_da_coluna])

# Ajusta o layout para evitar que os títulos e rótulos se sobreponham
plt.tight_layout()

plt.show()

In [0]:




# Seleciona o nome da coluna que será plotada
nome_da_coluna = variaveis_processo[1]

# Cria a figura com APENAS 1 subplot (eixo)
fig, ax = plt.subplots(figsize=(19, 8))

# --- Plot Único ---
# A coluna específica do seu DataFrame é passada aqui
sns.boxplot(data=X[nome_da_coluna], ax=ax, orient='v')
ax.set_title(f'Boxplot e Análise de Outliers para "{nome_da_coluna}"')
ax.set_ylabel('Distribuição dos dados')
ax.set_xlabel(nome_da_coluna)

# Chama a função para adicionar a caixa de estatísticas no plot
# A série de dados da sua coluna é passada como argumento
adicionar_caixa_estatisticas(ax, X[nome_da_coluna])

# Ajusta o layout para evitar que os títulos e rótulos se sobreponham
plt.tight_layout()

plt.show()

In [0]:




# Seleciona o nome da coluna que será plotada
nome_da_coluna = variaveis_processo[3]

# Cria a figura com APENAS 1 subplot (eixo)
fig, ax = plt.subplots(figsize=(19, 8))

# --- Plot Único ---
# A coluna específica do seu DataFrame é passada aqui
sns.boxplot(data=X[nome_da_coluna], ax=ax, orient='v')
ax.set_title(f'Boxplot e Análise de Outliers para "{nome_da_coluna}"')
ax.set_ylabel('Distribuição dos dados')
ax.set_xlabel(nome_da_coluna)

# Chama a função para adicionar a caixa de estatísticas no plot
# A série de dados da sua coluna é passada como argumento
adicionar_caixa_estatisticas(ax, X[nome_da_coluna])

# Ajusta o layout para evitar que os títulos e rótulos se sobreponham
plt.tight_layout()

plt.show()

Com isso podemos ter uma boa noção de como está a saúde dos dados.

Por fim uma análise tabular de como estão esses dados:

In [0]:
medest=MedidasEstatisticas(X,variaveis_processo,3)
medest

Graficamente temos para os outliers nivel 1:

In [0]:
medest=MedidasEstatisticas(X,variaveis_processo,3)
# 3) Valores da linha '%outlier1' nas colunas selecionadas
valores = medest.loc['%outlier1', variaveis_processo].astype(float)

# (Opcional) ordenar do maior para o menor
ordem = valores.sort_values(ascending=False).index
categorias = list(ordem)
valores = valores.loc[ordem]

# 4) Plot
fig, ax = plt.subplots(figsize=(16, 9))
bars = ax.bar(categorias, valores.values)

ax.set_xlabel('Variáveis de Processo')
ax.set_ylabel('% de outliers (1.5×IQR)')
ax.set_title('%Outliers por coluna ')
ax.set_xticklabels(categorias, rotation=90)

# margem no topo para não cortar os rótulos
ax.margins(y=0.10)

# Rótulos de dados (usa bar_label se disponível; caso contrário, fallback com annotate)
labels = [f"{v:.1f}%" for v in valores.values]
try:
    ax.bar_label(bars, labels=labels, padding=3, fontsize=9)
except AttributeError:
    for bar, lbl in zip(bars, labels):
        ax.annotate(
            lbl,
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            ha='center', va='bottom',
            xytext=(0, 3), textcoords='offset points',
            fontsize=9
        )

plt.tight_layout()

plt.show()

Para outliers de nível 2:

In [0]:
medest=MedidasEstatisticas(X,variaveis_processo,3)


# 3) Valores da linha '%outlier2' nas colunas selecionadas
valores = medest.loc['%outlier2', variaveis_processo].astype(float)

# (Opcional) ordenar do maior para o menor
ordem = valores.sort_values(ascending=False).index
categorias = list(ordem)
valores = valores.loc[ordem]

# 4) Plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(16, 9))
bars = ax.bar(categorias, valores.values)

ax.set_xlabel('Variáveis de Processo')
ax.set_ylabel('% de outliers (3×IQR)')
ax.set_title('%Outliers por coluna ')
ax.set_xticklabels(categorias, rotation=90)

# margem no topo para não cortar os rótulos
ax.margins(y=0.10)

# Rótulos de dados (usa bar_label se disponível; caso contrário, fallback com annotate)
labels = [f"{v:.1f}%" for v in valores.values]
try:
    ax.bar_label(bars, labels=labels, padding=3, fontsize=9)
except AttributeError:
    for bar, lbl in zip(bars, labels):
        ax.annotate(
            lbl,
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            ha='center', va='bottom',
            xytext=(0, 3), textcoords='offset points',
            fontsize=9
        )

plt.tight_layout()
plt.show()

### Dados Faltante

Antes de prosseguir em analisar qual técnicas devemos usar para lidar com dados faltantes devemos entender que existem três tipos de dados faltantes:
- Missing Completely at Random (MCAR): Ausente de Forma Totalmente Aleatória refere-se ao caso onde a ausência de um dado não tem nada haver com nenhum outra variável(feature) do seu dataset, nem com o próprio valor que está faltante.
- Missing at Random (MAR): Ausente de Forma Aleatória, a ausência está relacionada com outras features do dataset e não com a variável em sí.
- Missing Not at Random(MNAR): Ausente de forma não aleatória, a ausência do dado está diretamente realcionada ao valor que está faltando.


Para conseguirmos investigar cada uma dessas relações vamos analisar, uma coluna por vez nos dados faltantes, para isso iremos criar uma outra coluna para a coluna que está sendo analisada que conterá 1 se os dado for nulo e 0  caso contrário. E iremos dividir o dataset em dois analisado a distribuição de cada coluan restante para os dois grupos, verificando se são diferentes. Caso forem (caso forem significavamente diferente ) é um forte indício do tipo MAR.

In [0]:
variaveis_processo

In [0]:
dff = qtdColunasNulas(X)
dff = dff.sort_values(by=0,axis=1,ascending=False)#Ordena em Ordem Decrescente
#Plotando colunas vázias
fig, ax = plt.subplots(figsize=(19, 9))
bars = ax.bar(dff.columns, dff.values[0], color='deepskyblue')
ax.set_ylabel('Missing Values')
ax.tick_params(axis='x', labelrotation=90)#Rotaciona para aparecer o nome das colunas

# Rótulos de valor em cada barra
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.1, int(yval),
            ha='center', va='bottom')

ax.set_title('Colunas com Valores faltantes')

plt.show()

In [0]:
dff = X.copy()
nome_coluna_original =variaveis_processo[20] #[US3] Teor de carvão

# Define o nome da nova coluna indicadora
nome_coluna_indicador = f"{nome_coluna_original}_indicator"

#Cria uma coluna para informar qual linah está com valores faltantes
dff[nome_coluna_indicador] = dff[nome_coluna_original].isnull().astype(int)

#Separa em dois dataframes
dff_faltante = dff[dff[nome_coluna_indicador] == 1]
dff_completo = dff[dff[nome_coluna_indicador] == 0]
print(f"Quantidade de linhas faltantes coluna {nome_coluna_original}:{dff_faltante.shape[0]}\n")
print(f"Quantidade de linhas não faltantes coluna{nome_coluna_original}:{dff_completo.shape[0]}\n")

Aplicação do Teste de Hipótese:
- Hipótese Nula(H0): Os parâmetros comparados vem da mesma distribuição de dados.
- Hipótese Alternativa(H1): Não vem das mesma distribuição de dados.

Se  p-value for menos que 0.05 então rejeitamos H0.

In [0]:
variaveis_processo = [variavel for variavel in variaveis_processo if variavel != variaveis_processo[20] and variavel != 'Campanha']

In [0]:
import numpy as np
from scipy.stats import mannwhitneyu
s = 0
t = 0
for coluna in variaveis_processo[0:]:
    
    U, p_value = mannwhitneyu(dff_completo[coluna].dropna(), dff_faltante[coluna].dropna())
   
    if p_value <0.05:
        print(f"{coluna}, p-value:{p_value}, Existe Diferença significativa")
        s=s+1
    else:
        print(f"{coluna}, p-value:{p_value}, Não existe Diferença significativa")
    
    t=t+1
    
print(f'\n\n % de colunas com p-value menor que 0.05:{(s/t)*100:0.2f} %(Diferença Estatística)')
   

### Principal Componente Anlysis

Para conseguirmos aplicar PCA, aplicar os seguintes tratamentos de dados:
- Remover coluna Y,Legenda Y, Pô do sol,Nascer do Sol, Data, Ano.
- Dummie Encoding: Campanha (One Hot Encoding).

In [0]:
X = (X.drop(['Y', 'Pôr do sol', 'Nascer do sol', 'Data','Dia?'], axis=1)
      .pipe(pd.get_dummies, columns=['Campanha'],dtype=int))
X 

In [0]:
# Supondo que sua coluna original esteja em X['Legenda Y']
# Vamos criar uma array numpy alinhada com os dados
y_labels = X['Legenda Y'].apply(lambda x: 'Sem emissão' if x == 'Sem emissão' else 'Com emissão').values

# Definindo cores para facilitar a visualização
# Sem emissão = Azul, Com emissão = Vermelho
color_map = {'Sem emissão': 'tab:blue', 'Com emissão': 'tab:red'}

In [0]:
Xpca = X.drop(columns=['Legenda Y'],inplace=True)

In [0]:
X

In [0]:
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# Certifique-se de que 'dz' é o seu DataFrame original com dados faltantes

# 1. SEPARAR AS COLUNAS
# Seleciona apenas as colunas numéricas para a imputação
colunas_numericas = X.select_dtypes(include=['number']).columns
df_numerico = X[colunas_numericas]

# Guarda as colunas não-numéricas para juntar depois
colunas_nao_numericas = X.select_dtypes(exclude=['number']).columns
df_nao_numerico = X[colunas_nao_numericas]

# 2. APLICAR A IMPUTAÇÃO APENAS NOS DADOS NUMÉRICOS
mice_imputer = IterativeImputer(max_iter=10, random_state=42)

# O fit_transform é aplicado SOMENTE no dataframe numérico
imputed_data_numerico = mice_imputer.fit_transform(df_numerico)

# Converte o resultado (array numpy) de volta para um DataFrame, mantendo os nomes E O ÍNDICE
df_numerico_imputado = pd.DataFrame(
    imputed_data_numerico, 
    columns=df_numerico.columns, 
    index=X.index  # Mantém o índice original do dz
)

# 3. JUNTAR OS DATAFRAMES
# Concatena as colunas não-numéricas com as numéricas agora imputadas
df_final_completo = pd.concat([df_nao_numerico, df_numerico_imputado], axis=1)

# Reordena as colunas para a ordem original
df_final_completo = df_final_completo[X.columns]

# ---
# AGORA VOCÊ PODE RODAR O PCA USANDO 'df_final_completo'
# Exemplo: pca.fit(df_final_completo[colunas_numericas])

In [0]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
# 2. Padronização (Essencial para PCA)
# O PCA é sensível à escala, então é preciso colocar tudo na mesma grandeza
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_final_completo)

# 3. Instanciar e aplicar PCA
# n_components=2 reduzirá para 2 colunas. 
# Se quiser ver a variância ==total para decidir quantos manter, use n_components=None
pca = PCA(n_components=(len(df_final_completo.columns))) 
dados_reduzidos = pca.fit_transform(X_scaled)

# 4. Criando um DataFrame para exibir as variâncias de forma organizada
df_variancia = pd.DataFrame({
    'Componente': [f'PC{i+1}' for i in range(len(pca.explained_variance_ratio_))],
    'Variança Explicada': pca.explained_variance_ratio_,
    'Variança Acumulada': pca.explained_variance_ratio_.cumsum()
})

# Formatar para porcentagem para facilitar a leitura
df_variancia['Variança Explicada (%)'] = (df_variancia['Variança Explicada'] * 100).round(2)
df_variancia['Variança Acumulada (%)'] = (df_variancia['Variança Acumulada'] * 100).round(2)

print("Tabela de Variância:")
print(df_variancia[['Componente', 'Variança Explicada (%)', 'Variança Acumulada (%)']])

print(f"\nTotal de informação retida com 2 componentes: {df_variancia['Variança Acumulada'].iloc[-1]*100:.2f}%")

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(35, 6))

# 1. Gráfico da Variância por Componente (Scree Plot)
plt.subplot(1, 2, 1)
plt.plot(df_variancia['Componente'], df_variancia['Variança Explicada (%)'], marker='o', linestyle='-')
plt.title('Variância Explicada por Componente')
plt.xlabel('Componente')
plt.ylabel('Variância (%)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

plt.legend()

plt.tight_layout()
plt.show()

In [0]:

plt.figure(figsize=(35, 6))
# 2. Gráfico da Variância Acumulada
plt.subplot(1, 2, 2)
plt.plot(df_variancia['Componente'], df_variancia['Variança Acumulada (%)'], marker='o', linestyle='-', color='orange')
plt.title('Variância Acumulada')
plt.xlabel('Componente')
plt.ylabel('Acumulada (%)')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)

# Linha de referência opcional (ex: 90% ou 95%)
plt.axhline(y=95, color='r', linestyle='--', linewidth=0.8, label='95%')
plt.legend()

plt.tight_layout()
plt.show()

Vamos analisar, os scores com as duas primeiras componentes:

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import math

def biplot_split(score, coeff, col_names, labels, filtro_emissao=None, n_plots=4, width=20, height_per_plot=10):
    """
    filtro_emissao: 
        0 -> Exibe apenas pontos 'Sem emissão'
        1 -> Exibe apenas pontos 'Com emissão'
        None -> Exibe ambos (Padrão)
    """
    
    # 1. Configurar a escala
    xs = score[:, 0]
    ys = score[:, 1]
    scalex = 1.0 / (xs.max() - xs.min())
    scaley = 1.0 / (ys.max() - ys.min())
    
    # 2. Definir quais categorias vamos plotar baseado no parametro filtro_emissao
    todas_categorias = np.unique(labels)
    
    if filtro_emissao == 0:
        categorias_para_plotar = ['Sem emissão']
    elif filtro_emissao == 1:
        categorias_para_plotar = ['Com emissão']
    else:
        categorias_para_plotar = todas_categorias # Exibe tudo

    # 3. Configuração dos subplots
    n_features = coeff.shape[0]
    chunk_size = math.ceil(n_features / n_plots)
    
    fig, axes = plt.subplots(n_plots, 1, figsize=(width, n_plots * height_per_plot))
    if n_plots == 1: axes = [axes]

    # Mapa de cores fixo
    palette = {'Sem emissão': 'tab:blue', 'Com emissão': 'tab:red'}

    for i in range(n_plots):
        ax = axes[i]
        
        # --- A. PLOTAR OS PONTOS (SCORES) FILTRADOS ---
        for cat in categorias_para_plotar:
            # Só tenta plotar se a categoria existir nos dados (segurança)
            if cat in todas_categorias:
                mask = (labels == cat)
                ax.scatter(xs[mask] * scalex, ys[mask] * scaley, 
                           c=palette.get(cat, 'gray'), 
                           label=cat, 
                           alpha=0.4, 
                           s=40)

        ax.legend(title="Status", loc='upper right', frameon=True, fontsize=12)
        
        # --- B. PLOTAR AS SETAS (LOADINGS) - Mantemos sempre visíveis para contexto ---
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, n_features)
        
        for j in range(start_idx, end_idx):
            ax.arrow(0, 0, coeff[j, 0], coeff[j, 1], color='black', alpha=0.8, 
                     head_width=0.015, head_length=0.02, linewidth=1.2)
            
            ax.text(coeff[j, 0] * 1.15, coeff[j, 1] * 1.15, col_names[j], 
                    color='black', ha='center', va='center', weight='bold', fontsize=11,
                    bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))

        # Ajustes estéticos
        ax.set_xlabel(f"PC1", fontsize=12)
        ax.set_ylabel(f"PC2", fontsize=12)
        ax.set_title(f"Biplot Parte {i+1} (Features {start_idx} a {end_idx-1})", fontsize=14, weight='bold')
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.axvline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.set_xlim(-0.8, 0.8) 
        ax.set_ylim(-0.8, 0.8)

    plt.tight_layout()
    plt.show()

# --- EXEMPLOS DE USO ---

# Opção A: Não passar nada (Exibe AMBOS)
# biplot_split(dados_reduzidos, loadings, df_final_completo.columns, y_labels, n_plots=4)

# Opção B: Passar 0 (Exibe APENAS 'Sem emissão' - Azul)
# biplot_split(dados_reduzidos, loadings, df_final_completo.columns, y_labels, filtro_emissao=0, n_plots=4)

# Opção C: Passar 1 (Exibe APENAS 'Com emissão' - Vermelho)
# ==============================================================================
# CORREÇÃO: DEFINIR A VARIÁVEL LOADINGS
# ==============================================================================
# Os components_ são os vetores de carga.
# Fazemos a Transposta (.T) para ficar no formato (n_features, n_componentes)
loadings = pca.components_.T 

# ==============================================================================
# AGORA PODE CHAMAR A FUNÇÃO
# ==============================================================================

# Opção C: Passar 1 (Exibe APENAS 'Com emissão' - Vermelho)
biplot_split(
    score=dados_reduzidos, 
    coeff=loadings,  # <--- Agora esta variável existe!
    col_names=df_final_completo.columns, 
    labels=y_labels, 
    filtro_emissao=1, 
    n_plots=8
)

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import math

def biplot_split(score, coeff, col_names, labels, filtro_emissao=None, n_plots=4, width=20, height_per_plot=10):
    """
    filtro_emissao: 
        0 -> Exibe apenas pontos 'Sem emissão'
        1 -> Exibe apenas pontos 'Com emissão'
        None -> Exibe ambos (Padrão)
    """
    
    # 1. Configurar a escala
    xs = score[:, 0]
    ys = score[:, 1]
    scalex = 1.0 / (xs.max() - xs.min())
    scaley = 1.0 / (ys.max() - ys.min())
    
    # 2. Definir quais categorias vamos plotar baseado no parametro filtro_emissao
    todas_categorias = np.unique(labels)
    
    if filtro_emissao == 0:
        categorias_para_plotar = ['Sem emissão']
    elif filtro_emissao == 1:
        categorias_para_plotar = ['Com emissão']
    else:
        categorias_para_plotar = todas_categorias # Exibe tudo

    # 3. Configuração dos subplots
    n_features = coeff.shape[0]
    chunk_size = math.ceil(n_features / n_plots)
    
    fig, axes = plt.subplots(n_plots, 1, figsize=(width, n_plots * height_per_plot))
    if n_plots == 1: axes = [axes]

    # Mapa de cores fixo
    palette = {'Sem emissão': 'tab:blue', 'Com emissão': 'tab:red'}

    for i in range(n_plots):
        ax = axes[i]
        
        # --- A. PLOTAR OS PONTOS (SCORES) FILTRADOS ---
        for cat in categorias_para_plotar:
            # Só tenta plotar se a categoria existir nos dados (segurança)
            if cat in todas_categorias:
                mask = (labels == cat)
                ax.scatter(xs[mask] * scalex, ys[mask] * scaley, 
                           c=palette.get(cat, 'gray'), 
                           label=cat, 
                           alpha=0.4, 
                           s=40)

        ax.legend(title="Status", loc='upper right', frameon=True, fontsize=12)
        
        # --- B. PLOTAR AS SETAS (LOADINGS) - Mantemos sempre visíveis para contexto ---
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, n_features)
        
        for j in range(start_idx, end_idx):
            ax.arrow(0, 0, coeff[j, 0], coeff[j, 1], color='black', alpha=0.8, 
                     head_width=0.015, head_length=0.02, linewidth=1.2)
            
            ax.text(coeff[j, 0] * 1.15, coeff[j, 1] * 1.15, col_names[j], 
                    color='black', ha='center', va='center', weight='bold', fontsize=11,
                    bbox=dict(facecolor='white', alpha=0.6, edgecolor='none', pad=1))

        # Ajustes estéticos
        ax.set_xlabel(f"PC1", fontsize=12)
        ax.set_ylabel(f"PC2", fontsize=12)
        ax.set_title(f"Biplot Parte {i+1} (Features {start_idx} a {end_idx-1})", fontsize=14, weight='bold')
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.axvline(0, color='grey', linewidth=0.8, linestyle='--')
        ax.set_xlim(-0.8, 0.8) 
        ax.set_ylim(-0.8, 0.8)

    plt.tight_layout()
    plt.show()

# --- EXEMPLOS DE USO ---

# Opção A: Não passar nada (Exibe AMBOS)
# biplot_split(dados_reduzidos, loadings, df_final_completo.columns, y_labels, n_plots=4)

# Opção B: Passar 0 (Exibe APENAS 'Sem emissão' - Azul)
# biplot_split(dados_reduzidos, loadings, df_final_completo.columns, y_labels, filtro_emissao=0, n_plots=4)

# Opção C: Passar 1 (Exibe APENAS 'Com emissão' - Vermelho)
biplot_split(dados_reduzidos, loadings, df_final_completo.columns, y_labels, filtro_emissao=0, n_plots=8)